In [1]:
import sys; sys.path.append("../src")
import numpy as np
import pandas as pd
from xgboost import XGBClassifier

from data_prep import load_merged, temporal_split, numeric_features
from eval_harness import evaluate, pick_threshold_for_budget

df = load_merged("../data/raw")
train, val, test = temporal_split(df)
train, val = train.copy(), val.copy()   # we'll be adding columns; avoid slice warnings

y_train, y_val = train["isFraud"], val["isFraud"]


In [6]:
cat_cols = df.select_dtypes(include=["object", "str"]).columns.tolist()
print(f"{len(cat_cols)} categorical columns discarded until now:\n")
card = df[cat_cols].nunique().sort_values(ascending=False)
miss = df[cat_cols].isna().mean().round(2)
print(pd.DataFrame({"n_unique": card, "missing_frac": miss}))


31 categorical columns discarded until now:

               n_unique  missing_frac
DeviceInfo         1786          0.80
DeviceType            2          0.76
M1                    2          0.46
M2                    2          0.46
M3                    2          0.46
M4                    3          0.48
M5                    2          0.59
M6                    2          0.29
M7                    2          0.59
M8                    2          0.59
M9                    2          0.59
P_emaildomain        59          0.16
ProductCD             5          0.00
R_emaildomain        60          0.77
card4                 4          0.00
card6                 4          0.00
id_12                 2          0.76
id_15                 3          0.76
id_16                 2          0.78
id_23                 3          0.99
id_27                 2          0.99
id_28                 2          0.76
id_29                 2          0.76
id_30                75          0.87
id_31

In [7]:
def fit_categories(train_df, cols):
    """Learn the category vocabulary FROM TRAIN ONLY."""
    return {c: pd.CategoricalDtype(train_df[c].dropna().unique()) for c in cols}

def apply_categories(df_, vocab):
    """Cast using train's vocabulary. Values never seen in train -> NaN."""
    out = df_.copy()
    for c, dtype in vocab.items():
        out[c] = pd.Categorical(out[c], categories=dtype.categories)   # unseen values coerce to NaN
    return out

vocab = fit_categories(train, cat_cols)
train = apply_categories(train, vocab)
val   = apply_categories(val, vocab)

# how much of val falls outside train's vocabulary? (drift, quantified)
unseen = {c: val[c].isna().mean() - df.loc[val.index, c].isna().mean()
          for c in ["DeviceInfo", "id_31", "P_emaildomain"]}
print("fraction of val values unseen in train:", {k: round(v, 3) for k, v in unseen.items()})


fraction of val values unseen in train: {'DeviceInfo': np.float64(0.005), 'id_31': np.float64(0.069), 'P_emaildomain': np.float64(0.0)}


In [5]:
feats_num = numeric_features(train).tolist()
feats_all = feats_num + cat_cols

model_cat = XGBClassifier(tree_method="hist", eval_metric="logloss",
                          n_jobs=-1, random_state=42,
                          enable_categorical=True)          # ← the one new flag
model_cat.fit(train[feats_all], y_train)
proba_cat = model_cat.predict_proba(val[feats_all])[:, 1]

results = [
    {"name": "control: numeric-only (nb 03)", "pr_auc": 0.5571, "roc_auc": 0.9050},
    evaluate(y_val, proba_cat, name="+ categoricals (native)"),
]
pd.DataFrame(results).set_index("name")


,pr_auc,roc_auc,threshold,recall,precision,flagged,missed_fraud
name,,,,,,,
control: numeric-only (nb 03),0.5571,0.9050,NaN,NaN,NaN,NaN,NaN
+ categoricals (native),0.5738,0.9107,0.5,0.393,0.776,2335.0,2799.0


In [8]:
for part in (train, val):
    part["hour"] = (part["TransactionDT"] // 3600) % 24     # hour of day, 0-23
    part["dow"]  = (part["TransactionDT"] // 86400) % 7     # day of week, 0-6

# fraud rate by hour
print(train.groupby("hour")["isFraud"].mean().round(4))


hour
0     0.0310
1     0.0323
2     0.0345
3     0.0344
4     0.0476
5     0.0572
6     0.0656
7     0.0845
8     0.0920
9     0.1044
10    0.0727
11    0.0390
12    0.0301
13    0.0202
14    0.0241
15    0.0241
16    0.0297
17    0.0315
18    0.0327
19    0.0328
20    0.0317
21    0.0333
22    0.0319
23    0.0377
Name: isFraud, dtype: float64


In [9]:
# fit: per-card1 amount statistics FROM TRAIN ONLY
card_stats = (train.groupby("card1")["TransactionAmt"]
                   .agg(card1_amt_mean="mean", card1_amt_std="std"))

def add_amount_context(part, stats):
    out = part.join(stats, on="card1")               # unseen card1 in val -> NaN
    out["amt_ratio_card1"] = out["TransactionAmt"] / out["card1_amt_mean"]
    out["amt_z_card1"] = ((out["TransactionAmt"] - out["card1_amt_mean"])
                          / out["card1_amt_std"])
    out["amt_cents"] = out["TransactionAmt"] % 1     # decimal part — odd-cents signal
    return out

train = add_amount_context(train, card_stats)
val   = add_amount_context(val, card_stats)


In [11]:
ratio_cols = ["amt_ratio_card1", "amt_z_card1"]
for part in (train, val):
    part[ratio_cols] = part[ratio_cols].replace([np.inf, -np.inf], np.nan)

# sanity check: no more inf anywhere in the new columns
print("inf left in train:", np.isinf(train[ratio_cols]).sum().sum())
print("inf left in val:  ", np.isinf(val[ratio_cols]).sum().sum())


inf left in train: 0
inf left in val:   0


In [12]:
feats_time   = feats_all + ["hour", "dow"]
feats_amount = feats_time + ["card1_amt_mean", "card1_amt_std",
                             "amt_ratio_card1", "amt_z_card1", "amt_cents"]

for name, cols in [("+ time", feats_time), ("+ amount context", feats_amount)]:
    m = XGBClassifier(tree_method="hist", eval_metric="logloss", n_jobs=-1,
                      random_state=42, enable_categorical=True)
    m.fit(train[cols], y_train)
    results.append(evaluate(y_val, m.predict_proba(val[cols])[:, 1], name=name))

pd.DataFrame(results).set_index("name")


,pr_auc,roc_auc,threshold,recall,precision,flagged,missed_fraud
name,,,,,,,
control: numeric-only (nb 03),0.5571,0.9050,NaN,NaN,NaN,NaN,NaN
+ categoricals (native),0.5738,0.9107,0.5,0.3930,0.7760,2335.0,2799.0
+ time,0.5770,0.9073,0.5,0.3869,0.7894,2260.0,2827.0
+ time,0.5770,0.9073,0.5,0.3869,0.7894,2260.0,2827.0
+ amount context,0.5702,0.9081,0.5,0.4003,0.7756,2380.0,2765.0


In [ ]:
## Findings

Cumulative ablation on the temporal harness (val PR-AUC):

| Feature set | PR-AUC | Δ |
|---|---|---|
| numeric-only (control) | 0.557 | — |
| + 31 categoricals (native XGBoost) | 0.574 | **+0.017** |
| + hour/day-of-week | 0.577 | +0.003 |
| + per-card1 amount context | 0.570 | **−0.007 (rejected)** |

1. **Restoring categoricals is the single biggest win of the project so far** — more than
   all imbalance handling combined (nb 03). New information > reshuffled information.
2. **Time features added little despite a 5× hourly fraud-rate swing** — the model could
   already express the pattern by splitting raw `TransactionDT`. A derived feature only
   helps if the model couldn't reach its information already.
3. **Per-card amount statistics hurt validation performance** — thousands of card1 levels
   mean many stats are estimated from a handful of rows; the trees trust noise.
   Rejected and documented rather than silently kept.
4. Feature-level drift measured: 6.9% of val's `id_31` (browser) values never occur in
   train. Unseen-category → NaN policy handles this at serving time.
5. Engineering hazards hit and fixed: divide-by-zero `inf` from zero-variance cards
   (XGBoost accepts NaN, rejects inf); non-idempotent notebook cells duplicating results.

**Model going forward: numeric + categoricals + time, val PR-AUC 0.577.**
